# Project training configuration for local reproducible fine-tuning.

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
BASE_MODEL_NAME = "indobenchmark/indobert-base-p1"
TRAIN_PATH = "./train_data/train.csv"
VAL_PATH = "./train_data/val.csv"
MODEL_OUTPUT_DIR = "./model"
TOKENIZER_OUTPUT_DIR = "./tokenizer"
MAX_LENGTH = 128
NUM_LABELS = 2
TEXT_COL = "text"
LABEL_COL = "Label"

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# PII masking utilities applied before preprocessing and tokenization.

In [ ]:
PHONE_PATTERN = re.compile(r'(\+62|62|0)8[1-9][0-9]{6,10}')
EMAIL_PATTERN = re.compile(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}')
NIK_PATTERN = re.compile(r'\b\d{16}\b')

def mask_pii(text: str) -> str:
    if not isinstance(text, str):
        text = str(text)
    text = EMAIL_PATTERN.sub("<EMAIL>", text)
    text = PHONE_PATTERN.sub("<PHONE>", text)
    text = NIK_PATTERN.sub("<NIK>", text)
    return text

# Core text normalization for Indonesian social-media comments.

In [ ]:
URL_PATTERN = re.compile(r'https?://\S+|www\.\S+')
MENTION_PATTERN = re.compile(r'@\w+')
HASHTAG_PATTERN = re.compile(r'#\w+')
MULTISPACE_PATTERN = re.compile(r'\s+')

def preprocess_text(text: str) -> str:
    if not isinstance(text, str):
        text = str(text)
    text = text.lower().strip()
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

required_cols = {TEXT_COL, LABEL_COL}
if not required_cols.issubset(train_df.columns) or not required_cols.issubset(val_df.columns):
    raise ValueError(f"Dataset must contain columns: {required_cols}")

for df in [train_df, val_df]:
    df[TEXT_COL] = df[TEXT_COL].astype(str).apply(mask_pii).apply(preprocess_text)
    df[LABEL_COL] = df[LABEL_COL].astype(int)

print("Train size:", len(train_df))
print("Val size:", len(val_df))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, local_files_only=False)

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

train_ds = Dataset.from_pandas(train_df[[TEXT_COL, LABEL_COL]])
val_ds = Dataset.from_pandas(val_df[[TEXT_COL, LABEL_COL]])

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)

train_ds = train_ds.rename_column(LABEL_COL, "labels")
val_ds = val_ds.rename_column(LABEL_COL, "labels")

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=NUM_LABELS,
    local_files_only=False
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="binary")
    }

use_cuda = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir="./outputs",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=use_cuda
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()
eval_result = trainer.evaluate()
print("Evaluation:", eval_result)

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(TOKENIZER_OUTPUT_DIR, exist_ok=True)

trainer.model.save_pretrained(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(TOKENIZER_OUTPUT_DIR)

print("Saved model to:", MODEL_OUTPUT_DIR)
print("Saved tokenizer to:", TOKENIZER_OUTPUT_DIR)